# ExifTool Metadata Proof of Concept (POC)

This notebook demonstrates how to apply **ExifTool** to an image file (`.jfif`, `.jpg`, `.png`) and inspect the extracted metadata.

## What this POC does

1. Selects an image file from this project folder.
2. Finds `exiftool` on your machine.
3. Runs ExifTool and requests JSON output.
4. Displays all metadata and a short OSINT-focused subset.

## Prerequisite

Install ExifTool and make sure `exiftool` is available in your PATH.
- Windows: install ExifTool and ensure the executable can be called as `exiftool` in a terminal.
- Verify in terminal: `exiftool -ver`

In [1]:
from pathlib import Path
from typing import Optional
import shutil
import subprocess
import json

# Point this to any .jfif/.jpg/.png file in your workspace.
IMAGE_PATH = Path('cat.jfif')

print(f"Image target: {IMAGE_PATH.resolve()}")
print(f"Exists: {IMAGE_PATH.exists()}")

Image target: C:\Users\ZeroRanger967\OneDrive\Documents\Illinois Tech\Cyber Security Technologies\Group OSINT Project\cat.jfif
Exists: True


In [2]:
def find_exiftool() -> Optional[str]:
    """Return a callable exiftool command path if found."""
    candidates = [
        shutil.which('exiftool'),
        shutil.which('exiftool.exe'),
        r'C:\\exiftool\\exiftool.exe',
        r'C:\\Tools\\exiftool.exe',
    ]
    for candidate in candidates:
        if candidate and Path(candidate).exists():
            return str(candidate)
    return None

EXIFTOOL_CMD = find_exiftool()

if EXIFTOOL_CMD is None:
    raise FileNotFoundError(
        'ExifTool not found. Install it and ensure exiftool is in PATH, then rerun.'
    )

print(f'Using ExifTool: {EXIFTOOL_CMD}')

Using ExifTool: .\exiftool.EXE


In [3]:
if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Image not found: {IMAGE_PATH}')

result = subprocess.run(
    [EXIFTOOL_CMD, '-j', str(IMAGE_PATH)],
    capture_output=True,
    text=True,
    check=True,
)

records = json.loads(result.stdout)
metadata = records[0] if records else {}

print(f"Total metadata fields: {len(metadata)}")
metadata

Total metadata fields: 24


{'SourceFile': 'cat.jfif',
 'ExifToolVersion': 13.57,
 'FileName': 'cat.jfif',
 'Directory': '.',
 'FileSize': '29 kB',
 'FileModifyDate': '2026:04:05 18:59:19-05:00',
 'FileAccessDate': '2026:04:23 02:34:46-05:00',
 'FileCreateDate': '2026:04:21 18:50:07-05:00',
 'FilePermissions': '-rw-rw-rw-',
 'FileType': 'JPEG',
 'FileTypeExtension': 'jpg',
 'MIMEType': 'image/jpeg',
 'JFIFVersion': 1.01,
 'ResolutionUnit': 'inches',
 'XResolution': 72,
 'YResolution': 72,
 'ImageWidth': 593,
 'ImageHeight': 517,
 'EncodingProcess': 'Progressive DCT, Huffman coding',
 'BitsPerSample': 8,
 'ColorComponents': 3,
 'YCbCrSubSampling': 'YCbCr4:2:0 (2 2)',
 'ImageSize': '593x517',
 'Megapixels': 0.307}

In [4]:
# OSINT-relevant fields often checked in image metadata.
key_fields = [
    'FileName',
    'FileType',
    'MIMEType',
    'ImageWidth',
    'ImageHeight',
    'Make',
    'Model',
    'CreateDate',
    'DateTimeOriginal',
    'ModifyDate',
    'GPSLatitude',
    'GPSLongitude',
    'GPSPosition',
    'Artist',
    'Copyright',
    'Software',
]

summary = {k: metadata.get(k) for k in key_fields if k in metadata}
summary

{'FileName': 'cat.jfif',
 'FileType': 'JPEG',
 'MIMEType': 'image/jpeg',
 'ImageWidth': 593,
 'ImageHeight': 517}

In [5]:
# Optional: print grouped output exactly like CLI analysts often use.
cli_view = subprocess.run(
    [EXIFTOOL_CMD, '-a', '-u', '-g1', str(IMAGE_PATH)],
    capture_output=True,
    text=True,
    check=True,
)

print(cli_view.stdout[:4000])  # Truncate for notebook readability.

---- ExifTool ----
ExifTool Version Number         : 13.57
---- System ----
File Name                       : cat.jfif
Directory                       : .
File Size                       : 29 kB
File Modification Date/Time     : 2026:04:05 18:59:19-05:00
File Access Date/Time           : 2026:04:23 02:34:46-05:00
File Creation Date/Time         : 2026:04:21 18:50:07-05:00
File Permissions                : -rw-rw-rw-
---- File ----
File Type                       : JPEG
File Type Extension             : jpg
MIME Type                       : image/jpeg
Image Width                     : 593
Image Height                    : 517
Encoding Process                : Progressive DCT, Huffman coding
Bits Per Sample                 : 8
Color Components                : 3
Y Cb Cr Sub Sampling            : YCbCr4:2:0 (2 2)
---- JFIF ----
JFIF Version                    : 1.01
Resolution Unit                 : inches
X Resolution                    : 72
Y Resolution                    : 72
---- Comp

## Reddit API POC: Download 5 News Images to Session

This section fetches top posts from a news subreddit and downloads up to 5 image posts.
It supports two modes:

1. PRAW mode (optional): uses Reddit app credentials if provided.
2. Public mode (no credentials): uses Reddit's public JSON listing read-only.

### Setup notes

1. Recommended install (if missing):

```powershell
pip install praw requests
```

2. Credentials are optional for this notebook.
   - If you provide `REDDIT_CLIENT_ID` (and optionally `REDDIT_CLIENT_SECRET`), it will use PRAW.
   - If you skip credentials, it will use unauthenticated public JSON mode.
3. Optional environment variables:
   - `REDDIT_CLIENT_ID`
   - `REDDIT_CLIENT_SECRET`
   - `REDDIT_USER_AGENT`

In [6]:
# Run once per kernel session.
%pip install praw requests


   ---------------------------------------- 0/4 [websocket-client]
   ---------------------------------------- 0/4 [websocket-client]
   ---------------------------------------- 0/4 [websocket-client]
   ---------------------------------------- 0/4 [websocket-client]
   ---------------------------------------- 0/4 [websocket-client]
   -------------------- ------------------- 2/4 [prawcore]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ------------------------------ --------- 3/4 [praw]
   ---------------------------------------- 4/4 [praw]

Note: you may need to restart the kernel to use update

In [7]:
import os
from getpass import getpass
from pathlib import Path

import requests

print('[Cell 9] Starting Reddit client setup...')

try:
    import praw
    print('[Cell 9] Imported praw successfully.')
except ModuleNotFoundError:
    praw = None
    print('[Cell 9] praw is not installed; PRAW mode will be unavailable.')

# Optional Reddit app credentials. If absent, the notebook falls back to public JSON endpoints.
REDDIT_CLIENT_ID = os.getenv('REDDIT_CLIENT_ID') or ''
REDDIT_CLIENT_SECRET = os.getenv('REDDIT_CLIENT_SECRET') or ''
REDDIT_USER_AGENT = os.getenv('REDDIT_USER_AGENT') or 'osint-poc-public-readonly/1.0 by notebook'

print(f"[Cell 9] REDDIT_CLIENT_ID present: {bool(REDDIT_CLIENT_ID)}")
print(f"[Cell 9] REDDIT_CLIENT_SECRET present: {bool(REDDIT_CLIENT_SECRET)}")
print(f"[Cell 9] USER_AGENT: {REDDIT_USER_AGENT}")

if not REDDIT_CLIENT_ID:
    REDDIT_CLIENT_ID = input('Optional REDDIT_CLIENT_ID (press Enter to skip): ').strip()
if REDDIT_CLIENT_ID and not REDDIT_CLIENT_SECRET:
    REDDIT_CLIENT_SECRET = getpass('Optional REDDIT_CLIENT_SECRET (press Enter to skip): ').strip()

reddit = None
using_praw = False

if praw is not None and REDDIT_CLIENT_ID:
    print('[Cell 9] Attempting PRAW read-only initialization...')
    try:
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET or None,
            user_agent=REDDIT_USER_AGENT,
        )
        reddit.read_only = True
        using_praw = True
        print('[Cell 9] Reddit client initialized with PRAW (read-only).')
    except Exception as exc:
        print(f'[Cell 9] PRAW initialization failed; falling back to public JSON mode: {exc}')
else:
    if praw is None:
        print('[Cell 9] Using public JSON mode because praw is unavailable.')
    else:
        print('[Cell 9] Using public JSON mode because no credentials were provided.')

HEADERS = {'User-Agent': REDDIT_USER_AGENT}
print(f"[Cell 9] Setup complete. using_praw={using_praw}")

[Cell 9] Starting Reddit client setup...
[Cell 9] Imported praw successfully.
[Cell 9] REDDIT_CLIENT_ID present: False
[Cell 9] REDDIT_CLIENT_SECRET present: False
[Cell 9] USER_AGENT: osint-poc-public-readonly/1.0 by notebook
[Cell 9] Using public JSON mode because no credentials were provided.
[Cell 9] Setup complete. using_praw=False


In [8]:
NEWS_SUBREDDIT = 'news'       # Change to 'worldnews' if you prefer
TIME_FILTER = 'day'           # hour, day, week, month, year, all
POST_SCAN_LIMIT = 75          # Scan enough top posts to find image links
TARGET_IMAGE_COUNT = 5
SESSION_DIR = Path('Session')
SESSION_DIR.mkdir(parents=True, exist_ok=True)

print('[Cell 10] Starting post discovery...')
print(
    f"[Cell 10] Config: subreddit={NEWS_SUBREDDIT}, time_filter={TIME_FILTER}, "
    f"scan_limit={POST_SCAN_LIMIT}, target_images={TARGET_IMAGE_COUNT}"
)

def is_image_url(url: str) -> bool:
    lower = (url or '').lower()
    return lower.endswith(('.jpg', '.jpeg', '.png', '.jfif', '.webp'))

def clean_filename(name: str) -> str:
    keep = []
    for ch in name:
        keep.append(ch if ch.isalnum() or ch in (' ', '-', '_') else '_')
    return ''.join(keep).strip().replace(' ', '_')

def extract_image_from_post_data(data: dict) -> str:
    # 1) Direct image URL post
    candidate_url = data.get('url_overridden_by_dest') or data.get('url') or ''
    if is_image_url(candidate_url):
        return candidate_url

    # 2) Preview image for link/image posts
    preview = data.get('preview') or {}
    images = preview.get('images') or []
    if images:
        src = (images[0].get('source') or {}).get('url') or ''
        if src:
            return src.replace('&amp;', '&')

    # 3) Gallery media metadata
    if data.get('is_gallery') and data.get('media_metadata'):
        media_metadata = data.get('media_metadata') or {}
        for _, meta in media_metadata.items():
            src = (meta.get('s') or {}).get('u') or ''
            if src:
                return src.replace('&amp;', '&')

    return ''

def fetch_post_detail_image(post_id: str) -> str:
    if not post_id:
        return ''
    detail_url = f"https://www.reddit.com/comments/{post_id}.json"
    try:
        detail_resp = requests.get(detail_url, headers=HEADERS, params={'raw_json': 1}, timeout=20)
        detail_resp.raise_for_status()
        detail_data = detail_resp.json()[0]['data']['children'][0]['data']
        return extract_image_from_post_data(detail_data)
    except Exception as exc:
        print(f"[Cell 10] Detail lookup failed for {post_id}: {exc}")
        return ''

selected = []

if using_praw and reddit is not None:
    print('[Cell 10] Mode: PRAW read-only.')
    try:
        sub = reddit.subreddit(NEWS_SUBREDDIT)

        for post in sub.top(time_filter=TIME_FILTER, limit=POST_SCAN_LIMIT):
            if post.is_self:
                continue

            source_url = post.url or ''
            image_url = source_url if is_image_url(source_url) else ''

            if not image_url:
                image_url = fetch_post_detail_image(post.id)

            if not image_url:
                continue

            selected.append(
                {
                    'id': post.id,
                    'title': post.title,
                    'url': image_url,
                    'source_url': source_url,
                    'score': post.score,
                }
            )
            print(f"[Cell 10] Selected via PRAW: {post.id} :: image={image_url}")

            if len(selected) >= TARGET_IMAGE_COUNT:
                break
    except Exception as exc:
        print(f'[Cell 10] Error while collecting posts with PRAW: {exc}')
        raise
else:
    # Public, unauthenticated fallback using Reddit JSON listing.
    print('[Cell 10] Mode: Public JSON read-only.')
    listing_url = f"https://www.reddit.com/r/{NEWS_SUBREDDIT}/top.json"
    params = {'t': TIME_FILTER, 'limit': POST_SCAN_LIMIT, 'raw_json': 1}
    print(f"[Cell 10] Request URL: {listing_url}")
    print(f"[Cell 10] Request params: {params}")

    try:
        resp = requests.get(listing_url, headers=HEADERS, params=params, timeout=20)
        print(f"[Cell 10] HTTP status: {resp.status_code}")
        resp.raise_for_status()
        payload = resp.json()
    except Exception as exc:
        print(f'[Cell 10] Error during Reddit JSON request/parsing: {exc}')
        raise

    children = payload.get('data', {}).get('children', [])
    print(f"[Cell 10] Candidate posts returned: {len(children)}")

    for child in children:
        data = child.get('data', {})
        if data.get('is_self'):
            continue

        post_id = data.get('id', '')
        source_url = data.get('url_overridden_by_dest') or data.get('url') or ''
        image_url = extract_image_from_post_data(data)

        # Important fallback for /r/news link posts: listing may omit preview image.
        if not image_url:
            image_url = fetch_post_detail_image(post_id)

        if not image_url:
            continue

        selected.append(
            {
                'id': post_id,
                'title': data.get('title', ''),
                'url': image_url,
                'source_url': source_url,
                'score': data.get('score', 0),
            }
        )
        print(
            f"[Cell 10] Selected via JSON: {post_id} :: "
            f"source={source_url} :: image={image_url}"
        )

        if len(selected) >= TARGET_IMAGE_COUNT:
            break

print(f"[Cell 10] Found {len(selected)} image posts from r/{NEWS_SUBREDDIT} top {TIME_FILTER}.")
selected

[Cell 10] Starting post discovery...
[Cell 10] Config: subreddit=news, time_filter=day, scan_limit=75, target_images=5
[Cell 10] Mode: Public JSON read-only.
[Cell 10] Request URL: https://www.reddit.com/r/news/top.json
[Cell 10] Request params: {'t': 'day', 'limit': 75, 'raw_json': 1}
[Cell 10] HTTP status: 200
[Cell 10] Candidate posts returned: 22
[Cell 10] Selected via JSON: 1ssxq9z :: source=https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html :: image=https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7
[Cell 10] Selected via JSON: 1ssfxhd :: source=https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments :: image=https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A54.jpeg?auto=webp&s=4022c111561bace6f00e72c4936c4e812761c2e0
[Cell 10] Selected via JSON: 1ssukwq :: source=https://apnews.com/article/west-virginia-chemical-leak-aa346adf

[{'id': '1ssxq9z',
  'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'url': 'https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7',
  'source_url': 'https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
  'score': 11943},
 {'id': '1ssfxhd',
  'title': 'Texas can require public schools to display Ten Commandments, court rules | Texas',
  'url': 'https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A54.jpeg?auto=webp&s=4022c111561bace6f00e72c4936c4e812761c2e0',
  'source_url': 'https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments',
  'score': 10272},
 {'id': '1ssukwq',
  'title': 'Chemical leak at a West Virginia plant kills 2 people and sends 19 more to hospital, officials say',
  'url': 'https://external-preview.redd.it/bIozUX6kr_7nwcgzbyqP0Ts6WGv8-t7apxHppI8bg5Y.png?auto=webp&s=9

In [9]:
downloaded = []
print(f"[Cell 11] Starting downloads for {len(selected)} selected posts...")

for idx, item in enumerate(selected, start=1):
    url = item['url']
    source_url = item.get('source_url', '')
    print(f"[Cell 11] [{idx}] Download attempt: {url}")
    if source_url and source_url != url:
        print(f"[Cell 11] [{idx}] Source post URL: {source_url}")

    try:
        response = requests.get(url, timeout=20)
        print(f"[Cell 11] [{idx}] HTTP status: {response.status_code}")
        response.raise_for_status()
    except Exception as exc:
        print(f"[Cell 11] [{idx}] Skip (request failed): {url} :: {exc}")
        continue

    content_type = response.headers.get('Content-Type', '').lower()
    if 'image' not in content_type and not is_image_url(url):
        print(f"[Cell 11] [{idx}] Skip (not an image response): {url} :: content-type={content_type}")
        continue

    suffix = Path(url.split('?')[0]).suffix.lower()
    if suffix not in ('.jpg', '.jpeg', '.png', '.jfif', '.webp'):
        if 'png' in content_type:
            suffix = '.png'
        elif 'webp' in content_type:
            suffix = '.webp'
        elif 'jpeg' in content_type or 'jpg' in content_type:
            suffix = '.jpg'
        else:
            suffix = '.jpg'

    base_name = clean_filename(f"{idx:02d}_{item['id']}_{item['title'][:50]}")
    out_path = SESSION_DIR / f"{base_name}{suffix}"

    try:
        with open(out_path, 'wb') as f:
            f.write(response.content)
    except Exception as exc:
        print(f"[Cell 11] [{idx}] Failed to write file {out_path}: {exc}")
        continue

    downloaded.append(
        {
            'file': str(out_path),
            'title': item['title'],
            'source_url': source_url or url,
            'image_url': url,
            'score': item['score'],
        }
    )
    print(f"[Cell 11] [{idx}] Saved: {out_path}")

print(f"[Cell 11] Downloaded {len(downloaded)} images to: {SESSION_DIR.resolve()}")
downloaded

[Cell 11] Starting downloads for 5 selected posts...
[Cell 11] [1] Download attempt: https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7
[Cell 11] [1] Source post URL: https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html
[Cell 11] [1] HTTP status: 200
[Cell 11] [1] Saved: Session\01_1ssxq9z_Kalshi_suspends__fines_3_congressional_candidates.jpeg
[Cell 11] [2] Download attempt: https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A54.jpeg?auto=webp&s=4022c111561bace6f00e72c4936c4e812761c2e0
[Cell 11] [2] Source post URL: https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments
[Cell 11] [2] HTTP status: 200
[Cell 11] [2] Saved: Session\02_1ssfxhd_Texas_can_require_public_schools_to_display_Ten_Co.jpeg
[Cell 11] [3] Download attempt: https://external-preview.redd.it/bIozUX6kr_7nwcgzbyqP0Ts6WGv8-t7apxHppI8bg5Y.png?auto=webp&s=9efaf75c0343d26472

[{'file': 'Session\\01_1ssxq9z_Kalshi_suspends__fines_3_congressional_candidates.jpeg',
  'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'source_url': 'https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
  'image_url': 'https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7',
  'score': 11943},
 {'file': 'Session\\02_1ssfxhd_Texas_can_require_public_schools_to_display_Ten_Co.jpeg',
  'title': 'Texas can require public schools to display Ten Commandments, court rules | Texas',
  'source_url': 'https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments',
  'image_url': 'https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A54.jpeg?auto=webp&s=4022c111561bace6f00e72c4936c4e812761c2e0',
  'score': 10272},
 {'file': 'Session\\04_1ssh8qn_Armed_men_rob_Brinks_armored_truck_in_Northeast_Ph.jpe

In [10]:
# Export artifacts for reverse-image workflows.
import csv
import json

if not downloaded:
    raise RuntimeError('No downloaded images found. Run Cell 10 and Cell 11 first.')

manifest_csv = SESSION_DIR / 'reverse_image_manifest.csv'
manifest_json = SESSION_DIR / 'reverse_image_manifest.json'
urls_txt = SESSION_DIR / 'reverse_image_urls.txt'
files_txt = SESSION_DIR / 'reverse_image_files.txt'

rows = []
for item in downloaded:
    row = {
        'file': item.get('file', ''),
        'image_url': item.get('image_url', item.get('url', '')),
        'source_url': item.get('source_url', ''),
        'title': item.get('title', ''),
        'score': item.get('score', 0),
    }
    rows.append(row)

with open(manifest_csv, 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(
        f,
        fieldnames=['file', 'image_url', 'source_url', 'title', 'score']
    )
    writer.writeheader()
    writer.writerows(rows)

with open(manifest_json, 'w', encoding='utf-8') as f:
    json.dump(rows, f, ensure_ascii=False, indent=2)

with open(urls_txt, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(r['image_url'] + '\n')

with open(files_txt, 'w', encoding='utf-8') as f:
    for r in rows:
        f.write(r['file'] + '\n')

print(f"Wrote CSV manifest: {manifest_csv}")
print(f"Wrote JSON manifest: {manifest_json}")
print(f"Wrote image URL list: {urls_txt}")
print(f"Wrote local file list: {files_txt}")
print(f"Total records: {len(rows)}")

# Display quick preview of what to feed into reverse image tooling.
rows[:3]

Wrote CSV manifest: Session\reverse_image_manifest.csv
Wrote JSON manifest: Session\reverse_image_manifest.json
Wrote image URL list: Session\reverse_image_urls.txt
Wrote local file list: Session\reverse_image_files.txt
Total records: 4


[{'file': 'Session\\01_1ssxq9z_Kalshi_suspends__fines_3_congressional_candidates.jpeg',
  'image_url': 'https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7',
  'source_url': 'https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
  'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'score': 11943},
 {'file': 'Session\\02_1ssfxhd_Texas_can_require_public_schools_to_display_Ten_Co.jpeg',
  'image_url': 'https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A54.jpeg?auto=webp&s=4022c111561bace6f00e72c4936c4e812761c2e0',
  'source_url': 'https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments',
  'title': 'Texas can require public schools to display Ten Commandments, court rules | Texas',
  'score': 10272},
 {'file': 'Session\\04_1ssh8qn_Armed_men_rob_Brinks_armored_truck_in_Northeast_Ph.jpe

In [60]:
# Feasibility test: run one reverse image search query using the local helper.
%pip install -q beautifulsoup4

import importlib.util
from pathlib import Path

module_path = SESSION_DIR / 'reverse_image_search.py'

if not module_path.exists():
    raise RuntimeError(f'Missing helper module: {module_path}')

spec = importlib.util.spec_from_file_location('reverse_image_search', str(module_path))
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
GoogleReverseImageSearch = module.GoogleReverseImageSearch

if not downloaded:
    raise RuntimeError('No downloaded records found. Run Cells 10-12 first.')

test_image_url = downloaded[0].get('image_url') or downloaded[0].get('url')
print(f"Testing reverse image search with URL: {test_image_url}")

request = GoogleReverseImageSearch()
response = request.response(
    query='OSINT reverse image check',
    image_url=test_image_url,
    max_results=3,
    fallback_result=downloaded[0]
 )

print(f"Response type: {type(response).__name__}")
response

Note: you may need to restart the kernel to use updated packages.
Testing reverse image search with URL: https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7
Response type: SearchResults


In [58]:
# Check whether reverse-image response includes a similarity score.
score_fields = {'score', 'similarity', 'similarity_score', 'confidence', 'match_score'}

if isinstance(response, str):
    print('Reverse search returned a status string, not structured matches:')
    print(response)
    print('No similarity score is available in this response.')
else:
    rows = getattr(response, 'results', [])
    print(f'Reverse search result count: {len(rows)}')
    if rows:
        sample_keys = set(rows[0].keys())
        found = sorted(k for k in sample_keys if k.lower() in score_fields or 'score' in k.lower() or 'similar' in k.lower())
        print('Sample keys in result rows:', sorted(sample_keys))
        if found:
            print('Potential score-like fields found:', found)
        else:
            print('No similarity score fields found. Library currently returns title/link only.')
    else:
        print('No match rows to inspect.')

Reverse search result count: 1
Sample keys in result rows: ['link', 'match_type', 'source', 'title']
No similarity score fields found. Library currently returns title/link only.


In [22]:
# 3) Ollama news-integrity scorer (API-first, CLI fallback)
import subprocess
import json
import os
import re
import shutil
import requests


def _get_ollama_cmd():
    # Try PATH first, then common Windows install location.
    exe = shutil.which('ollama')
    if exe:
        return exe
    fallback = os.path.expandvars(r"%LOCALAPPDATA%\Programs\Ollama\ollama.exe")
    if os.path.exists(fallback):
        return fallback
    return None


def _clamp_int(value, lower, upper, default=0):
    try:
        parsed = int(value)
    except (TypeError, ValueError):
        parsed = default
    return max(lower, min(upper, parsed))


def _sanitize_ollama_output(text):
    # Remove ANSI escape/control sequences and non-printable control chars.
    text = re.sub(r'\x1B\[[0-?]*[ -/]*[@-~]', '', text)
    text = re.sub(r'\x1B[@-_]', '', text)
    text = text.replace('\r', '')
    # Keep newline and tab; remove other C0 controls.
    text = ''.join(ch for ch in text if ch in ('\n', '\t') or ord(ch) >= 32)
    return text


def _extract_json_from_output(raw_output):
    clean_output = _sanitize_ollama_output(raw_output)

    # 1) Direct JSON parse
    try:
        obj = json.loads(clean_output)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass

    # 2) Strip fenced code blocks if present
    text = clean_output.replace('```json', '```').replace('```JSON', '```')
    if '```' in text:
        parts = text.split('```')
        # odd indices are fenced segments
        for i in range(1, len(parts), 2):
            candidate = parts[i].strip()
            try:
                obj = json.loads(candidate)
                if isinstance(obj, dict):
                    return obj
            except Exception:
                continue

    # 3) Scan for JSON object substrings using raw decoder
    decoder = json.JSONDecoder()
    for idx, ch in enumerate(clean_output):
        if ch != '{':
            continue
        try:
            obj, _end = decoder.raw_decode(clean_output[idx:])
            if isinstance(obj, dict):
                return obj
        except Exception:
            continue

    return None


def _build_news_integrity_prompt(article_record, reverse_matches=None, exif_summary=None):
    reverse_matches = reverse_matches or []
    payload = {
        'article_title': article_record.get('title', ''),
        'article_source_url': article_record.get('source_url', ''),
        'image_url': article_record.get('image_url', article_record.get('url', '')),
        'reverse_image_matches': reverse_matches,
        'exif_summary': exif_summary or {},
    }

    return f"""
You are a digital forensic analyst evaluating news integrity.

Task:
Assess whether the article image is likely authentic, misleading, manipulated, or used in a sensational/clickbait way.
Use ONLY the structured evidence provided.
Do not invent facts.

Return ONLY valid JSON with these keys:
- "image_manipulation_risk": integer 0-25
- "context_mismatch_risk": integer 0-25
- "sensationalism_risk": integer 0-20
- "source_credibility_risk": integer 0-15
- "evidence_consistency_risk": integer 0-15
- "integrity_risk_score": integer 0-100 (must equal sum of the 5 scores)
- "verdict": one of ["likely_authentic", "uncertain", "likely_misleading_or_clickbait"]
- "confidence": integer 0-100
- "reason": short explanation (<= 60 words)
- "red_flags": array of short strings (max 5)
- "supporting_signals": array of short strings (max 5)

Scoring guidance:
- image_manipulation_risk: signs of edits, synthetic artifacts, inconsistent forensic cues.
- context_mismatch_risk: image reused in unrelated events/time/location/topic.
- sensationalism_risk: emotionally loaded framing or headline-image mismatch for attention.
- source_credibility_risk: weak provenance, low-trust host pattern, missing corroboration.
- evidence_consistency_risk: contradictions between article claims, reverse hits, and EXIF clues.

Formula:
integrity_risk_score = image_manipulation_risk + context_mismatch_risk + sensationalism_risk + source_credibility_risk + evidence_consistency_risk

Evidence JSON:
{json.dumps(payload, ensure_ascii=False, indent=2)}
"""


def _run_ollama_api(model, prompt, timeout):
    # Non-streaming API avoids terminal control characters from CLI streaming output.
    r = requests.post(
        'http://127.0.0.1:11434/api/generate',
        json={
            'model': model,
            'prompt': prompt,
            'stream': False,
            'format': 'json',
        },
        timeout=timeout,
    )
    r.raise_for_status()
    payload = r.json()
    return payload.get('response', '')


def _run_ollama_cli(cmd, model, prompt, timeout):
    attempts = [
        [cmd, 'run', model, '--format', 'json'],
        [cmd, 'run', model],
    ]

    last_stdout = ''
    last_stderr = ''
    for args in attempts:
        result = subprocess.run(
            args,
            input=prompt.encode('utf-8'),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=timeout,
        )
        stdout = result.stdout.decode(errors='ignore').strip()
        stderr = result.stderr.decode(errors='ignore').strip()

        if stdout:
            return stdout, stderr

        last_stdout = stdout
        last_stderr = stderr

    return last_stdout, last_stderr


def llm_news_integrity_score(article_record, reverse_matches=None, exif_summary=None, ollama_model='llama3'):
    prompt = _build_news_integrity_prompt(
        article_record=article_record,
        reverse_matches=reverse_matches,
        exif_summary=exif_summary,
    )

    raw_stderr = ''
    raw_output = ''

    # Try local Ollama HTTP API first.
    try:
        raw_output = _run_ollama_api(ollama_model, prompt, timeout=240)
    except Exception as api_exc:
        raw_stderr = f'API path failed: {api_exc}'
        cmd = _get_ollama_cmd()
        if cmd is None:
            return {
                'integrity_risk_score': 50,
                'verdict': 'uncertain',
                'reason': 'Ollama API unreachable and CLI not found',
                'raw_stderr': raw_stderr,
            }
        try:
            raw_output, cli_stderr = _run_ollama_cli(cmd, ollama_model, prompt, timeout=240)
            raw_stderr = f'{raw_stderr} | CLI stderr: {cli_stderr}'.strip(' |')
        except subprocess.TimeoutExpired:
            return {'integrity_risk_score': 50, 'verdict': 'uncertain', 'reason': 'LLM call timed out'}

    parsed = _extract_json_from_output(raw_output)
    if parsed is None:
        return {
            'integrity_risk_score': 50,
            'verdict': 'uncertain',
            'reason': 'LLM output not parseable',
            'raw_output': raw_output,
            'raw_stderr': raw_stderr,
        }

    components = {
        'image_manipulation_risk': _clamp_int(parsed.get('image_manipulation_risk', 0), 0, 25),
        'context_mismatch_risk': _clamp_int(parsed.get('context_mismatch_risk', 0), 0, 25),
        'sensationalism_risk': _clamp_int(parsed.get('sensationalism_risk', 0), 0, 20),
        'source_credibility_risk': _clamp_int(parsed.get('source_credibility_risk', 0), 0, 15),
        'evidence_consistency_risk': _clamp_int(parsed.get('evidence_consistency_risk', 0), 0, 15),
    }

    parsed.update(components)
    parsed['integrity_risk_score'] = sum(components.values())
    parsed['confidence'] = _clamp_int(parsed.get('confidence', 60), 0, 100, default=60)

    allowed_verdicts = {'likely_authentic', 'uncertain', 'likely_misleading_or_clickbait'}
    verdict = str(parsed.get('verdict', 'uncertain')).strip().lower()
    parsed['verdict'] = verdict if verdict in allowed_verdicts else 'uncertain'

    if 'reason' not in parsed:
        parsed['reason'] = 'LLM integrity scoring completed'

    return parsed


# 3.a) Pull Ollama model (runs when this cell is executed)
def pull_ollama_model(model_name='llama3'):
    cmd = _get_ollama_cmd()
    if cmd is None:
        raise RuntimeError('Ollama CLI not found; install Ollama and ensure it is on PATH')
    print(f'Pulling Ollama model: {model_name} (this will download the model when run)')
    subprocess.run([cmd, 'pull', model_name], check=True)

In [52]:
# 4) News integrity evaluator using article + reverse-image evidence
from urllib.parse import urlparse, parse_qs, unquote


NON_ARTICLE_DOMAINS = {
    'artofit.org', 'wallpapercave.com', 'wallpapersafari.com', 'patreon.com',
    'tumblr.com', 'twitter.com', 'x.com', 'facebook.com', 'instagram.com',
    'pinterest.com', 'youtube.com', 'tiktok.com'
}


def _unwrap_possible_redirect(url):
    if not url:
        return ''
    try:
        p = urlparse(url)
        if 'duckduckgo.com' in p.netloc and p.path.startswith('/l/'):
            qs = parse_qs(p.query)
            uddg = qs.get('uddg', [])
            if uddg:
                return unquote(uddg[0])
    except Exception:
        return url
    return url


def _canonicalize_url(url):
    raw = _unwrap_possible_redirect((url or '').strip())
    if not raw:
        return ''
    try:
        p = urlparse(raw)
    except Exception:
        return raw

    netloc = p.netloc.lower().replace('www.', '')
    path = (p.path or '/').rstrip('/') or '/'
    scheme = p.scheme.lower() if p.scheme else 'https'
    return f'{scheme}://{netloc}{path}'


def _normalize_domain(url):
    if not url:
        return ''
    return urlparse(_unwrap_possible_redirect(url)).netloc.lower().replace('www.', '')


def _is_same_article(source_url, candidate_url):
    src = _canonicalize_url(source_url)
    cand = _canonicalize_url(candidate_url)
    return bool(src and cand and src == cand)


def _is_article_like_link(link):
    domain = _normalize_domain(link)
    if not domain or domain in NON_ARTICLE_DOMAINS:
        return False

    path = (urlparse(_unwrap_possible_redirect(link)).path or '').lower()
    if path.endswith(('.jpg', '.jpeg', '.png', '.gif', '.webp')):
        return False

    article_markers = ['/news/', '/article/', '/world/', '/us/', '/202', '/story/']
    if any(marker in path for marker in article_markers):
        return True

    return len(path) > 20 and ('-' in path or '/' in path.strip('/'))


def _is_valid_reverse_match(item, match_row):
    link = match_row.get('link', '')
    if not link:
        return False
    if match_row.get('match_type') == 'source_baseline':
        return False
    if _is_same_article(item.get('source_url', ''), link):
        return False
    return _is_article_like_link(link)


def _build_reverse_queries(item):
    queries = ['OSINT reverse image check']
    title = (item.get('title', '') or '').strip()
    if title:
        queries.append(title)
        short_title = ' '.join(title.split()[:10])
        if short_title and short_title != title:
            queries.append(short_title)

    source_domain = _normalize_domain(item.get('source_url', ''))
    if source_domain and title:
        queries.append(f'"{title}" -site:{source_domain}')

    seen = set()
    ordered = []
    for q in queries:
        if not q or q in seen:
            continue
        seen.add(q)
        ordered.append(q)
    return ordered


def _collect_live_reverse_matches(item):
    # Attempt fresh reverse-image lookup for each item when helper is available.
    if 'GoogleReverseImageSearch' not in globals():
        return []

    image_url = item.get('image_url', item.get('url', ''))
    if not image_url:
        return []

    all_matches = []
    queries = _build_reverse_queries(item)

    # Expand attempts progressively until we get at least one valid reverse match.
    for q in queries:
        for max_results in (8, 12, 16):
            try:
                req = GoogleReverseImageSearch()
                resp_obj = req.response(
                    query=q,
                    image_url=image_url,
                    max_results=max_results,
                    fallback_result=item,
                )
            except Exception:
                continue

            rows = getattr(resp_obj, 'results', []) if not isinstance(resp_obj, str) else []
            for row in rows:
                link = row.get('link', '')
                title = row.get('title', '')
                source = row.get('source', 'reverse-image')
                match_type = row.get('match_type', 'reverse_image')
                if not link:
                    continue

                # Drop same-article hits from provider results; source baseline is added later.
                if _is_same_article(item.get('source_url', ''), link):
                    continue

                all_matches.append({
                    'title': title,
                    'link': _unwrap_possible_redirect(link),
                    'source': source,
                    'match_type': match_type,
                })

            dedup = []
            seen = set()
            for m in all_matches:
                sig = (_canonicalize_url(m.get('link', '')), m.get('title', ''))
                if sig in seen:
                    continue
                seen.add(sig)
                dedup.append(m)
            all_matches = dedup

            if any(_is_valid_reverse_match(item, m) for m in all_matches):
                return all_matches

    return all_matches


def _collect_reverse_matches_for_item(item, item_index=0):
    matches = []

    # First preference: run fresh reverse-image lookup with expanded query strategy.
    matches.extend(_collect_live_reverse_matches(item))

    # Back-compat: if a prior reverse-image response exists from Cell 13, use it for item 1.
    if item_index == 0 and 'response' in globals():
        resp = globals().get('response')
        if hasattr(resp, 'results') and isinstance(resp.results, list):
            for row in resp.results[:5]:
                link = row.get('link', '')
                if _is_same_article(item.get('source_url', ''), link):
                    continue
                matches.append({
                    'title': row.get('title', ''),
                    'link': _unwrap_possible_redirect(link),
                    'source': row.get('source', 'reverse-image'),
                    'match_type': row.get('match_type', 'reverse_image'),
                })

    # Optional global map support.
    if 'reverse_matches_by_source' in globals():
        source_map = globals().get('reverse_matches_by_source') or {}
        key = item.get('source_url', '')
        if isinstance(source_map, dict) and key in source_map:
            for row in source_map.get(key, [])[:8]:
                link = row.get('link', '')
                if _is_same_article(item.get('source_url', ''), link):
                    continue
                matches.append({
                    'title': row.get('title', ''),
                    'link': _unwrap_possible_redirect(link),
                    'source': row.get('source', 'reverse-image-map'),
                    'match_type': row.get('match_type', 'reverse_image'),
                })

    # Always include the article itself as baseline provenance evidence.
    if item.get('source_url'):
        matches.append({
            'title': item.get('title', ''),
            'link': _canonicalize_url(item.get('source_url', '')),
            'source': 'article-source',
            'match_type': 'source_baseline',
        })

    dedup = []
    seen = set()
    for m in matches:
        sig = (_canonicalize_url(m.get('link', '')), m.get('title', ''))
        if sig in seen:
            continue
        seen.add(sig)
        dedup.append(m)

    return dedup[:20]


def _extract_external_article_matches(item, reverse_matches):
    source_domain = _normalize_domain(item.get('source_url', ''))
    external = []

    for m in reverse_matches:
        link = m.get('link', '')
        match_domain = _normalize_domain(link)
        if not match_domain or not source_domain:
            continue
        if match_domain == source_domain:
            continue
        if _is_same_article(item.get('source_url', ''), link):
            continue
        if not _is_article_like_link(link):
            continue

        external.append({
            'title': m.get('title', ''),
            'link': _canonicalize_url(link),
            'domain': match_domain,
            'source': m.get('source', ''),
            'match_type': m.get('match_type', ''),
        })

    dedup = []
    seen = set()
    for x in external:
        if x['link'] in seen:
            continue
        seen.add(x['link'])
        dedup.append(x)

    return dedup


def evaluate_news_integrity(item, item_index=0, use_llm=True, ollama_model='llama3'):
    reverse_matches = _collect_reverse_matches_for_item(item, item_index=item_index)
    external_article_matches = _extract_external_article_matches(item, reverse_matches)

    exif_summary = {
        'local_file': item.get('file', ''),
        'image_url': item.get('image_url', item.get('url', '')),
        'reverse_search_status': (
            'valid_match_found' if any(_is_valid_reverse_match(item, m) for m in reverse_matches)
            else 'no_valid_reverse_match'
        ),
    }

    llm_eval = (
        llm_news_integrity_score(
            article_record=item,
            reverse_matches=reverse_matches,
            exif_summary=exif_summary,
            ollama_model=ollama_model,
        )
        if use_llm
        else {'integrity_risk_score': 50, 'verdict': 'uncertain', 'reason': 'LLM scoring disabled'}
    )

    return {
        'title': item.get('title', ''),
        'source_url': item.get('source_url', ''),
        'source_domain': _normalize_domain(item.get('source_url', '')),
        'image_url': item.get('image_url', item.get('url', '')),
        'reverse_match_count': len(reverse_matches),
        'reverse_matches': reverse_matches,
        'has_external_article_match': bool(external_article_matches),
        'external_article_match_count': len(external_article_matches),
        'external_article_matches': external_article_matches[:10],
        'reverse_search_status': (
            'valid_match_found' if any(_is_valid_reverse_match(item, m) for m in reverse_matches)
            else 'no_valid_reverse_match'
        ),
        'integrity_reasoning': llm_eval,
        'integrity_risk_score': llm_eval.get('integrity_risk_score', 50),
        'verdict': llm_eval.get('verdict', 'uncertain'),
    }

In [61]:
# 4.b) Tighten valid reverse-match filtering for production
import re
from collections import Counter

NON_ARTICLE_DOMAINS.update({
    'linkedin.com', 'reddit.com', 'flickr.com', 'imdb.com',
    'wallpapers.com', 'genius.com',
})

SIMILARITY_THRESHOLD = 60
MIN_TITLE_TOKEN_OVERLAP = 0.12
MAX_SHARED_LINK_USAGE = 1
MAX_SHARED_DOMAIN_USAGE = 2


def _tokenize(text):
    tokens = re.findall(r'[a-z0-9]+', (text or '').lower())
    stop = {
        'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'on', 'for', 'with', 'by',
        'from', 'at', 'is', 'are', 'was', 'were', 'be', 'as', 'it', 'this', 'that',
        'new', 'says', 'say', 'after', 'over', 'under', 'into', 'about'
    }
    return [t for t in tokens if t not in stop and len(t) > 2]


def _title_overlap_ratio(article_title, match_title):
    a = set(_tokenize(article_title))
    b = set(_tokenize(match_title))
    if not a or not b:
        return 0.0
    inter = len(a.intersection(b))
    return inter / max(1, len(a))


def _is_article_like_link(link):
    domain = _normalize_domain(link)
    if not domain or domain in NON_ARTICLE_DOMAINS:
        return False

    path = (urlparse(_unwrap_possible_redirect(link)).path or '').lower()
    if path.endswith(('.jpg', '.jpeg', '.png', '.gif', '.webp')):
        return False

    # Require stronger article signals to reduce false positives.
    article_markers = ['/news/', '/article/', '/world/', '/us/', '/202', '/story/', '/feature/']
    if any(marker in path for marker in article_markers):
        return True

    slug_like = len(path) > 25 and '-' in path and path.count('/') >= 1
    return slug_like


def _is_valid_reverse_match(item, match_row):
    link = match_row.get('link', '')
    if not link:
        return False
    if match_row.get('match_type') == 'source_baseline':
        return False
    if _is_same_article(item.get('source_url', ''), link):
        return False
    if not _is_article_like_link(link):
        return False

    score = match_row.get('score')
    if score is None:
        return False
    try:
        if float(score) < float(SIMILARITY_THRESHOLD):
            return False
    except Exception:
        return False

    overlap = _title_overlap_ratio(item.get('title', ''), match_row.get('title', ''))
    if overlap < MIN_TITLE_TOKEN_OVERLAP:
        return False

    return True


def _extract_external_article_matches(item, reverse_matches):
    source_domain = _normalize_domain(item.get('source_url', ''))
    external = []

    for m in reverse_matches:
        link = m.get('link', '')
        match_domain = _normalize_domain(link)
        if not match_domain or not source_domain:
            continue
        if match_domain == source_domain:
            continue
        if _is_same_article(item.get('source_url', ''), link):
            continue
        if not _is_valid_reverse_match(item, m):
            continue

        overlap = _title_overlap_ratio(item.get('title', ''), m.get('title', ''))
        external.append({
            'title': m.get('title', ''),
            'link': _canonicalize_url(link),
            'domain': match_domain,
            'source': m.get('source', ''),
            'match_type': m.get('match_type', ''),
            'score': m.get('score'),
            'title_overlap': round(overlap, 3),
        })

    dedup = []
    seen = set()
    for x in external:
        if x['link'] in seen:
            continue
        seen.add(x['link'])
        dedup.append(x)

    return dedup

In [62]:
# 5) Run integrity analysis over collected articles
# Optional first run (uncomment if model is not available yet):
# pull_ollama_model('llama3')

if 'downloaded' not in globals() or not downloaded:
    raise RuntimeError('No downloaded records found. Run the article/image acquisition cells first.')

integrity_results = []
for idx, item in enumerate(downloaded):
    try:
        result = evaluate_news_integrity(
            item=item,
            item_index=idx,
            use_llm=True,
            ollama_model='llama3',
        )
    except Exception as exc:
        result = {
            'title': item.get('title', ''),
            'source_url': item.get('source_url', ''),
            'source_domain': '',
            'image_url': item.get('image_url', item.get('url', '')),
            'reverse_match_count': 0,
            'reverse_matches': [],
            'has_external_article_match': False,
            'external_article_match_count': 0,
            'external_article_matches': [],
            'reverse_search_status': 'no_valid_reverse_match',
            'similarity_threshold': SIMILARITY_THRESHOLD if 'SIMILARITY_THRESHOLD' in globals() else 60,
            'max_similarity_score': None,
            'integrity_reasoning': {'integrity_risk_score': 50, 'verdict': 'uncertain', 'reason': f'Execution error: {exc}'},
            'integrity_risk_score': 50,
            'verdict': 'uncertain',
        }

    all_scores = []
    for m in result.get('reverse_matches', []):
        s = m.get('score')
        if s is None:
            continue
        try:
            all_scores.append(float(s))
        except Exception:
            continue

    result['similarity_threshold'] = SIMILARITY_THRESHOLD if 'SIMILARITY_THRESHOLD' in globals() else 60
    result['max_similarity_score'] = max(all_scores) if all_scores else None

    integrity_results.append(result)

# Drop shared irrelevant matches across articles (same link reused across many items).
link_counts = Counter()
domain_counts = Counter()
for row in integrity_results:
    for m in row.get('external_article_matches', []):
        link_counts[m.get('link', '')] += 1
        domain_counts[m.get('domain', '')] += 1

for row in integrity_results:
    filtered = []
    for m in row.get('external_article_matches', []):
        if link_counts.get(m.get('link', ''), 0) > MAX_SHARED_LINK_USAGE:
            continue
        if domain_counts.get(m.get('domain', ''), 0) > MAX_SHARED_DOMAIN_USAGE:
            continue
        filtered.append(m)
    row['external_article_matches'] = filtered
    row['external_article_match_count'] = len(filtered)
    row['has_external_article_match'] = len(filtered) > 0

summary_rows = []
for r in integrity_results:
    reason = r.get('integrity_reasoning', {})
    summary_rows.append({
        'title': r.get('title', ''),
        'verdict': r.get('verdict', 'uncertain'),
        'integrity_risk_score': r.get('integrity_risk_score', 50),
        'reverse_match_count': r.get('reverse_match_count', 0),
        'external_article_match_count': r.get('external_article_match_count', 0),
        'has_external_article_match': r.get('has_external_article_match', False),
        'reverse_search_status': r.get('reverse_search_status', 'no_valid_reverse_match'),
        'max_similarity_score': r.get('max_similarity_score'),
        'similarity_threshold': r.get('similarity_threshold', 60),
        'reason': reason.get('reason', ''),
    })

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows).sort_values('integrity_risk_score', ascending=False))
except Exception:
    print(summary_rows)

# Detailed output for downstream pipeline steps
integrity_results

,title,verdict,integrity_risk_score,reverse_match_count,external_article_match_count,has_external_article_match,reverse_search_status,max_similarity_score,similarity_threshold,reason
2,Armed men rob Brinks armored truck in Northeas...,likely_misleading_or_clickbait,58,19,0,False,no_valid_reverse_match,None,60,Multiple reverse image matches with unrelated ...
0,"Kalshi suspends, fines 3 congressional candida...",uncertain,50,1,0,False,no_valid_reverse_match,None,60,Inconsistent reverse image matches and no EXIF...
1,Texas can require public schools to display Te...,likely_misleading_or_clickbait,45,17,0,False,no_valid_reverse_match,None,60,Multiple reverse image matches suggest the ima...
3,California sees lowest number of firearm-relat...,uncertain,35,17,0,False,no_valid_reverse_match,None,60,Multiple reverse image matches unrelated to th...


[{'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'source_url': 'https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
  'source_domain': 'cnbc.com',
  'image_url': 'https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7',
  'reverse_match_count': 1,
  'reverse_matches': [{'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
    'link': 'https://cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
    'source': 'article-source',
    'match_type': 'source_baseline'}],
  'has_external_article_match': False,
  'external_article_match_count': 0,
  'external_article_matches': [],
  'reverse_search_status': 'no_valid_reverse_match',
  'integrity_reasoning': {'image_manipulation_risk': 5,
   'context_mismatch_risk': 15,
   'sensationalism_risk': 10,
   'source_credibility_risk': 8,
   '

In [19]:
# Debug: inspect raw Ollama output when parsing fails
for i, row in enumerate(integrity_results[:2], start=1):
    reasoning = row.get('integrity_reasoning', {})
    print(f'Item {i} title:', row.get('title', '')[:90])
    print('reason:', reasoning.get('reason'))
    ro = reasoning.get('raw_output', '')
    rs = reasoning.get('raw_stderr', '')
    print('raw_output_len:', len(ro))
    print('raw_stderr_len:', len(rs))
    if ro:
        print('raw_output_snippet:', ro[:500])
    if rs:
        print('raw_stderr_snippet:', rs[:500])
    print('-' * 80)

Item 1 title: Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions
reason: LLM output not parseable
raw_output_len: 510
raw_stderr_len: 2003
raw_output_snippet: {
  "image_manipulation_risk": 5,
  "context_mismatch_risk": 10,
  "sensationalism_risk": 8,
  "source_credibility_risk": 2,
  "evidence_consistency_risk": 3,
  "integrity_risk_score": 28,
  "verdict": "uncertain",
  "confidence": 60,
  "reason": "Image may have been used in a previous article, and the sensat
sensationalized title may be intended to grab attention.",
  "red_flags": ["reused image", "sensationalized title"],
  "supporting_signals": ["manifest-fallback source", "exif data m
raw_stderr_snippet: ⠙ ⠹ ⠸ ⠼ [?
--------------------------------------------------------------------------------
Item 2 title: Texas can require public schools to display Ten Commandments, court rules | Texas
reason: LLM output not parseable
raw_output_len: 517
raw_stderr_len: 1956
raw_output_snippet: {
  

In [24]:
# Confirm whether reverse-image matches point to other articles/domains
from urllib.parse import urlparse

cross_article_summary = []
for row in integrity_results:
    source_url = row.get('source_url', '')
    source_domain = urlparse(source_url).netloc.lower().replace('www.', '') if source_url else ''

    external_links = []
    for m in row.get('reverse_matches', []):
        link = m.get('link', '')
        if not link:
            continue
        match_domain = urlparse(link).netloc.lower().replace('www.', '')
        if match_domain and source_domain and match_domain != source_domain:
            external_links.append({'link': link, 'domain': match_domain, 'source': m.get('source', '')})

    cross_article_summary.append({
        'title': row.get('title', ''),
        'source_domain': source_domain,
        'external_match_count': len(external_links),
        'external_matches': external_links[:5],
    })

try:
    import pandas as pd
    display(pd.DataFrame([
        {
            'title': x['title'],
            'source_domain': x['source_domain'],
            'external_match_count': x['external_match_count'],
        }
        for x in cross_article_summary
    ]))
except Exception:
    print(cross_article_summary)

cross_article_summary

,title,source_domain,external_match_count
0,"Kalshi suspends, fines 3 congressional candida...",cnbc.com,0
1,Texas can require public schools to display Te...,theguardian.com,0
2,Armed men rob Brinks armored truck in Northeas...,nbcphiladelphia.com,0
3,California sees lowest number of firearm-relat...,abc7.com,0


[{'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'source_domain': 'cnbc.com',
  'external_match_count': 0,
  'external_matches': []},
 {'title': 'Texas can require public schools to display Ten Commandments, court rules | Texas',
  'source_domain': 'theguardian.com',
  'external_match_count': 0,
  'external_matches': []},
 {'title': 'Armed men rob Brinks armored truck in Northeast Philly',
  'source_domain': 'nbcphiladelphia.com',
  'external_match_count': 0,
  'external_matches': []},
 {'title': 'California sees lowest number of firearm-related deaths since 1968, new data shows',
  'source_domain': 'abc7.com',
  'external_match_count': 0,
  'external_matches': []}]

In [63]:
# JSON preview: explicit reverse-image and integrity fields
import json

json_preview = []
for r in integrity_results:
    json_preview.append({
        'title': r.get('title', ''),
        'source_url': r.get('source_url', ''),
        'image_url': r.get('image_url', ''),
        'has_external_article_match': r.get('has_external_article_match', False),
        'external_article_match_count': r.get('external_article_match_count', 0),
        'external_article_matches': r.get('external_article_matches', []),
        'reverse_search_status': r.get('reverse_search_status', 'no_valid_reverse_match'),
        'similarity_threshold': r.get('similarity_threshold', 60),
        'max_similarity_score': r.get('max_similarity_score'),
        'integrity_risk_score': r.get('integrity_risk_score', 50),
        'verdict': r.get('verdict', 'uncertain'),
        'reason': r.get('integrity_reasoning', {}).get('reason', ''),
    })

print(json.dumps(json_preview, ensure_ascii=False, indent=2)[:9000])
json_preview[:2]

[
  {
    "title": "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
    "source_url": "https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html",
    "image_url": "https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7",
    "has_external_article_match": false,
    "external_article_match_count": 0,
    "external_article_matches": [],
    "reverse_search_status": "no_valid_reverse_match",
    "similarity_threshold": 60,
    "max_similarity_score": null,
    "integrity_risk_score": 50,
    "verdict": "uncertain",
    "reason": "Inconsistent reverse image matches and no EXIF data correlation."
  },
  {
    "title": "Texas can require public schools to display Ten Commandments, court rules | Texas",
    "source_url": "https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments",
    "image_url": "https://external-preview.redd.it

[{'title': "Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions",
  'source_url': 'https://www.cnbc.com/2026/04/22/kalshi-insider-trading-congress.html',
  'image_url': 'https://external-preview.redd.it/rtljPEyxwFZ_oDqskcw08zsEJgPeG4VSHjcklA_9C3g.jpeg?auto=webp&s=7f0e6e44e54b96dc256c047eb1cf4c17b32c8eb7',
  'has_external_article_match': False,
  'external_article_match_count': 0,
  'external_article_matches': [],
  'reverse_search_status': 'no_valid_reverse_match',
  'similarity_threshold': 60,
  'max_similarity_score': None,
  'integrity_risk_score': 50,
  'verdict': 'uncertain',
  'reason': 'Inconsistent reverse image matches and no EXIF data correlation.'},
 {'title': 'Texas can require public schools to display Ten Commandments, court rules | Texas',
  'source_url': 'https://www.theguardian.com/us-news/2026/apr/21/texas-public-schools-ten-commandments',
  'image_url': 'https://external-preview.redd.it/MlrGaDR8SGBU5e5MU_VhCF0QgZqfNuUd0VTHbOw5A5

In [64]:
# Quick check: how many items have external article matches?
matched = [r for r in integrity_results if r.get('has_external_article_match')]
print('items_with_external_article_match =', len(matched))
for r in matched:
    print('-', r.get('title', '')[:90], '| count =', r.get('external_article_match_count', 0))
    for m in r.get('external_article_matches', [])[:3]:
        print('   ', m.get('domain', ''), '|', m.get('match_type', ''), '|', m.get('link', ''))

items_with_external_article_match = 0


In [50]:
# Sanity check: show top non-baseline reverse matches so we can inspect why external count is zero
for r in integrity_results:
    print('\nTITLE:', r.get('title', '')[:95])
    shown = 0
    for m in r.get('reverse_matches', []):
        if m.get('match_type') == 'source_baseline':
            continue
        print('  -', m.get('source', ''), '|', m.get('match_type', ''), '|', m.get('link', '')[:140])
        shown += 1
        if shown >= 5:
            break
    if shown == 0:
        print('  - no non-baseline matches')


TITLE: Kalshi suspends, fines 3 congressional candidates in 'insider trading' enforcement actions
  - bing | reverse_image | https://www.artofit.org/image-gallery/304626362307335862/van-life-showering/
  - bing | reverse_image | https://wordsworthsmuse.tumblr.com/
  - bing | reverse_image | https://www.patreon.com/posts/breeze-83407492
  - bing | reverse_image | https://www.linkedin.com/posts/lifebites-global_light-on-your-feet-4-embracing-change-as-activity-7186860906738278401-HzU7
  - bing | reverse_image | https://www.reddit.com/r/cat/comments/1dcqvcn/i_lov_u_babe/

TITLE: Texas can require public schools to display Ten Commandments, court rules | Texas
  - bing | reverse_image | https://www.artofit.org/image-gallery/304626362307335862/van-life-showering/
  - bing | reverse_image | https://www.imdb.com/name/nm7380180/mediaviewer/rm1735699713
  - bing | reverse_image | https://www.hisuzanne.com/
  - bing | reverse_image | https://www.linkedin.com/posts/lifebites-global_light-on-your